#RNN

##Tokenizer

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
veri = [
    'Bu bir örnek cümle.',
    'Başka bir cümle de var.',
    'Farklı bir örnek metin.'
]

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(veri)
sayisal_karsilik = tokenizer.texts_to_sequences(veri)

In [ ]:
print("Kelime indeksleri : ",tokenizer.word_index)

Kelime indeksleri :  {'bir': 1, 'örnek': 2, 'cümle': 3, 'bu': 4, 'başka': 5, 'de': 6, 'var': 7, 'farklı': 8, 'metin': 9}


In [ ]:
for sayi in sayisal_karsilik:
  print(sayi)

[4, 1, 2, 3]
[5, 1, 3, 6, 7]
[8, 1, 2, 9]


In [ ]:
sabit_uzunluk = pad_sequences(sayisal_karsilik,maxlen=5)

In [ ]:
for sayi in sabit_uzunluk:
  print(sayi)

[0 4 1 2 3]
[5 1 3 6 7]
[0 8 1 2 9]


##Embedding

In [ ]:
from keras.layers import Embedding
import numpy as np

In [ ]:
giris_boyutu = len(tokenizer.word_index) + 1

boyut = 300

embedding_layer = Embedding(
    input_dim=giris_boyutu,
    output_dim=boyut,
    input_length=5
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
kimlikli_veri = embedding_layer(np.array(sabit_uzunluk))

In [ ]:
print("Embedding Katman Sonucu : ")
print(kimlikli_veri)

Embedding Katman Sonucu : 
tf.Tensor(
[[[ 2.2412363e-02 -1.5937198e-02  3.5775278e-02 ... -1.1523616e-02
   -3.5664141e-02 -1.9702649e-02]
  [ 2.9124808e-02  2.2078145e-02 -4.5939483e-02 ... -3.7327230e-02
   -1.6302358e-02  1.9635689e-02]
  [ 2.1201286e-02 -1.9030511e-02  2.2053268e-02 ...  3.2974210e-02
    1.7240968e-02  1.1049140e-02]
  [-4.1053392e-02 -1.9719377e-03  1.1915505e-02 ...  4.6609726e-02
    2.9368486e-02 -3.3775888e-02]
  [-5.0641522e-03 -8.2949176e-03  4.8127238e-02 ...  2.0801973e-02
   -4.9238037e-02  3.7977185e-02]]

 [[ 1.4938597e-02 -4.9751259e-02 -3.7062347e-02 ...  4.7039460e-02
    4.5429263e-02  2.4716232e-02]
  [ 2.1201286e-02 -1.9030511e-02  2.2053268e-02 ...  3.2974210e-02
    1.7240968e-02  1.1049140e-02]
  [-5.0641522e-03 -8.2949176e-03  4.8127238e-02 ...  2.0801973e-02
   -4.9238037e-02  3.7977185e-02]
  [ 1.9390669e-02 -4.4620801e-02 -6.6448003e-05 ... -1.1145793e-02
   -2.9259861e-02 -3.5740830e-02]
  [ 3.8494278e-02  1.0026574e-02  2.8810594e-02 ...

##RNN

In [ ]:
!pip install joblib
!pip install colorama

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from keras.layers import SimpleRNN, Dense
from keras.utils import to_categorical
import joblib
from colorama import Fore, Back, Style

In [ ]:
!pip install -q gdown
import gdown

gdown.download('https://drive.google.com/uc?id=1osVIYeq6JkyiuOiyrbw_WLuhblxUuvl-', 'oPhishing_Email.csv', quiet=False)

In [ ]:
model_yolu = 'rnn.h5'

In [ ]:
mail_verisi = pd.read_csv('Phishing_Email.csv')
mail_verisi = mail_verisi.where((pd.notnull(mail_verisi)),'')

In [ ]:
mail_verisi.loc[mail_verisi['Email Type']=='Safe Email','Email Type'] = 0
mail_verisi.loc[mail_verisi['Email Type']=='Phishing Email','Email Type'] = 1

In [ ]:
X = mail_verisi['Email Text']
y = mail_verisi['Email Type']

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(X)
indeksli_veri = tokenizer.texts_to_sequences(X)

In [ ]:
word_index = tokenizer.word_index
kelime_uzunlugu = 1000
sabit_uzunluk = pad_sequences(indeksli_veri, maxlen=kelime_uzunlugu)

In [ ]:
y = to_categorical(y)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(sabit_uzunluk,y, test_size=0.2, random_state=42)

In [ ]:
model = Sequential()
model.add(Embedding(len(word_index)+1, output_dim=128, input_length=kelime_uzunlugu))
model.add(SimpleRNN(64,activation='relu'))
model.add(Dense(2,activation='softmax'))
model.compile(optimizer='adam',loss='categorical_crossentropy',metrics=['accuracy'])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.fit(X_train,y_train, epochs=5, batch_size=32, validation_data=(X_test,y_test))

Epoch 1/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 42s 82ms/step - accuracy: 0.7703 - loss: 78.6710 - val_accuracy: 0.8504 - val_loss: 0.3435
Epoch 2/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 37s 78ms/step - accuracy: 0.9589 - loss: 0.1428 - val_accuracy: 0.9499 - val_loss: 0.1281
Epoch 3/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 36s 78ms/step - accuracy: 0.9848 - loss: 0.0411 - val_accuracy: 0.9525 - val_loss: 0.1235
Epoch 4/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 36s 78ms/step - accuracy: 0.9879 - loss: 0.0253 - val_accuracy: 0.9676 - val_loss: 0.0904
Epoch 5/5
467/467 ━━━━━━━━━━━━━━━━━━━━ 37s 79ms/step - accuracy: 0.9875 - loss: 0.0247 - val_accuracy: 0.9601 - val_loss: 0.1012


In [ ]:
model.save(model_yolu)

In [ ]:
import tensorflow as tf
model = tf.keras.models.load_model(model_yolu)

In [ ]:
while True:
  giris = input('Email mesajını giriniz : ')
  sayisal_kelime = tokenizer.texts_to_sequences([giris])
  sabit_uzunluk = pad_sequences(sayisal_kelime, maxlen=kelime_uzunlugu)
  tahmin = model.predict(sabit_uzunluk)
  if np.argmax(tahmin) == 1:
    print(Fore.Red, 'Bu bir kimlik avı emaili olabilir.')
  else:
    print(Fore.GREEN, 'Bu bir güvenli emaildir.')
  print(Fore.BLUE, 'Doğruluk puanı : ', tahmin)
